# TexNet streaming P detection + rapid magnitude updates

**EQViT-torch Phase 2** — reproducible, event-disjoint, latency-aware workflow.

> Research prototype: validate locally before operational EEW use.

## Operational concept

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from eqvit_torch.eew import extract_p_centered,mc_magnitude
from eqvit_torch import EQViTMagnitude
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; mag=EQViTMagnitude().to(DEVICE)
ck=torch.load('txed_mag_best.pt',map_location=DEVICE); mag.load_state_dict(ck['state_dict']); mag.eval()

In [ ]:
# stream: (N,3), 100 Hz, physical/count amplitude preserved for magnitude.
# p_sample: returned by EQCCT-torch trigger logic.
def magnitude_update(stream,p_sample,post_s=29,mc=20):
 n=int((1+post_s)*100); w=extract_p_centered(stream,p_sample,n=n,pre=100)
 # For the original model, pad to 3000 samples; shorter-latency models should be trained separately.
 if len(w)<3000: w=np.pad(w,((0,3000-len(w)),(0,0)))
 return mc_magnitude(mag,w,n=mc,device=DEVICE)

In [ ]:
# Do NOT interpret padded short windows as validated short-window EEW models.
# Train dedicated models at each horizon first.
horizons=[2,4,5,10,20,29]
print(pd.DataFrame({'post_P_seconds':horizons,'status':['train dedicated model']*len(horizons)}))

## Trigger de-duplication checklist